# 03 — Portfolio ledger and end-to-end flow

## Learning objectives

- Explain the frozen Phase C contracts, deterministic event order, and next-eligible-open timing.
- Trace a real 57/60 target intent through order, fill, costs, cash, position, valuation, snapshot, validation, and reconciliation.
- Independently check principal accounting invariants with `Decimal`.
- Distinguish behavior demonstrated by the accepted real run from behavior proven only by synthetic fixtures/tests.
- Trace what actually happens after a momentum definition changes, including the missing orchestration boundary.

**Evidence examined.** HEAD `00e35d98a49492a7913a1e862117c5ae19757d06`; Phase C `phasec-fa439d650410376aae9e`; reproduction under `phase_c/reproductions/00e35d9`; pinned GPW Phase B `phaseb-f88fc2d38e9811ed1573`.

Ending equity is an accounting/reconciliation checksum, not investment evidence.

## Configuration

        The accepted run and reproduction are immutable and read-only.

In [1]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd

REPO_ROOT = Path(os.environ.get("ATS_REPO_ROOT", r"D:\Stock\ATS"))
DATA_ROOT = Path(os.environ.get("ATS_DATA_ROOT", r"D:\Stock\data\ATS"))
PROJECT_ROOT = REPO_ROOT / "source" / "python"
RESEARCH_ROOT = REPO_ROOT / "RESEARCH"
GPW_MANIFEST = Path(os.environ.get(
    "ATS_GPW_MANIFEST",
    DATA_ROOT / "phase_b" / "versions" / "phaseb-f88fc2d38e9811ed1573" / "manifest.json",
))
US_MANIFEST = Path(os.environ.get(
    "ATS_US_MANIFEST",
    DATA_ROOT / "phase_b" / "versions" / "phaseb-5d7086751156ac48cef3" / "manifest.json",
))
PHASE_A_RUN = Path(os.environ.get(
    "ATS_PHASE_A_RUN", DATA_ROOT / "phase_a" / "runs" / "phasea-2a2b3898aba37814"
))
PHASE_A_EXTENDED_RUN = Path(os.environ.get(
    "ATS_PHASE_A_EXTENDED_RUN",
    DATA_ROOT / "decision_oriented_phase_a" / "runs" / "extension-20260820T163347Z",
))
PHASE_C_RUN = Path(os.environ.get(
    "ATS_PHASE_C_RUN", DATA_ROOT / "phase_c" / "runs" / "phasec-fa439d650410376aae9e"
))
PHASE_C_REPRODUCTION = Path(os.environ.get(
    "ATS_PHASE_C_REPRODUCTION",
    DATA_ROOT / "phase_c" / "reproductions" / "00e35d9" / "phasec-fa439d650410376aae9e",
))

src = str(PROJECT_ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)

native_library_dir = str(Path(sys.prefix) / "Library" / "bin")
path_entries = [entry.rstrip("\\").lower() for entry in os.environ.get("PATH", "").split(os.pathsep)]
if native_library_dir.rstrip("\\").lower() not in path_entries:
    raise RuntimeError(f"Conda native-library directory is absent from kernel PATH: {native_library_dir}")

required = [REPO_ROOT, DATA_ROOT, GPW_MANIFEST, PHASE_A_RUN, PHASE_C_RUN]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Required retained evidence is missing: {missing}")

pd.set_option("display.max_rows", 12)
pd.set_option("display.max_columns", 14)
pd.set_option("display.width", 140)
print({
    "repo": str(REPO_ROOT),
    "data": str(DATA_ROOT),
    "python": sys.version.split()[0],
    "native_library_path": True,
    "jupyter_runtime": os.environ.get("JUPYTER_RUNTIME_DIR"),
})

{'repo': 'D:\\Stock\\ATS', 'data': 'D:\\Stock\\data\\ATS', 'python': '3.12.13', 'native_library_path': True, 'jupyter_runtime': 'D:\\Stock\\ATS\\RESEARCH\\.tmp\\ats-env\\jupyter'}


## Validate the accepted run and inspect its contracts

Production validation re-hashes inputs/artifacts, parses every ledger contract, reconstructs order translation independently, re-reads canonical fill/valuation sources, and checks accounting/timing invariants. Reconciliation is recomputed; no standalone reconciliation report is retained.

In [2]:
from ats_portfolio.validation import validate_run, reconcile_run
from ats_contracts.portfolio import LedgerRunManifest

manifest = LedgerRunManifest.model_validate_json((PHASE_C_RUN / "manifest.json").read_text(encoding="utf-8"))
validation = validate_run(PHASE_C_RUN)
reconciliation = reconcile_run(PHASE_C_RUN)
print({
    "run_id": manifest.run_id,
    "implementation_commit": manifest.implementation_provenance["commit"],
    "provenance_clean": manifest.implementation_provenance["clean"],
    "phase_b_manifest_id": manifest.phase_b_manifest_id,
    "artifact_count": validation["artifact_count"],
    "fills": validation["accounting"]["fills"],
    "sessions": validation["accounting"]["sessions"],
    "reconciliation_hash": reconciliation["reconciliation_hash"],
})

{'run_id': 'phasec-fa439d650410376aae9e', 'implementation_commit': '00e35d98a49492a7913a1e862117c5ae19757d06', 'provenance_clean': True, 'phase_b_manifest_id': 'phaseb-f88fc2d38e9811ed1573', 'artifact_count': 16, 'fills': 5, 'sessions': 11, 'reconciliation_hash': '5d20048cccf2e6a1ff94990fd44eca622b694192f0dd5e2f8bea4a95bf0e7136'}


**Interpretation.** The final run is the only retained Phase C run with clean provenance at documentation-complete HEAD and a matching reproduction. There is no explicit accepted pointer, so this acceptance is evidence-based inference. Earlier immutable runs remain retained but are superseded as completion evidence.

## Contracts and deterministic event order

`TargetWeightIntent` freezes decision time, information availability, eligible execution time, source manifest, official/usable/eligible counts, exclusions, weights, and provenance. Unknown schemas/enums/identities, duplicate IDs/revisions, mutable manifest pointers, inconsistent counts, and negative/over-one batch weights fail closed.

Per session the engine performs:

1. record beginning state;
2. at modeled open, apply eligible known security/corporate events;
3. reveal only the session open;
4. translate eligible target batches to orders;
5. execute sells before buys, with slippage/commission and explicit movements;
6. reveal completed-bar fields at bar completion;
7. value positions, enforce invariants, and emit the snapshot.

Same-close execution is absent. `information_available_ts <= decision_ts < eligible_open` is required. Missing opens produce rejection/deferment, never fill or zero/forward fill.

## Load retained ledgers through production readers

The accepted integration spans 11 sessions, five fills, and two held securities. We focus on the first batch and its explicit unavailable LOTOS negative control.

In [3]:
from ats_portfolio.storage import read_intents, read_ledger

intents = read_intents(PHASE_C_RUN / "ledgers" / "intents.csv")
orders = read_ledger(PHASE_C_RUN / "ledgers" / "orders.csv", "orders")
fills = read_ledger(PHASE_C_RUN / "ledgers" / "fills.csv", "fills")
cash_movements = read_ledger(PHASE_C_RUN / "ledgers" / "cash_movements.csv", "cash_movements")
position_movements = read_ledger(PHASE_C_RUN / "ledgers" / "position_movements.csv", "position_movements")
valuations = read_ledger(PHASE_C_RUN / "ledgers" / "valuations.csv", "valuations")
snapshots = read_ledger(PHASE_C_RUN / "ledgers" / "portfolio_snapshots.csv", "portfolio_snapshots")
rejections = read_ledger(PHASE_C_RUN / "ledgers" / "rejections.csv", "rejections")

first_batch = [intent for intent in intents if intent.batch_id == "gpw-ref-20201127"]
display(pd.DataFrame([{
    "intent_id": item.intent_id,
    "security_id": item.security_id,
    "target_weight": str(item.target_weight),
    "information_available_ts": item.information_available_ts,
    "decision_ts": item.decision_ts,
    "eligible_execution_ts": item.earliest_eligible_execution_ts,
    "official/usable/eligible": f"{item.official_universe_denominator}/{item.usable_price_count}/{item.feature_eligible_count}",
} for item in first_batch]))
display(pd.DataFrame([row.model_dump(mode="json") for row in rejections]).head(5))

,intent_id,security_id,target_weight,information_available_ts,decision_ts,eligible_execution_ts,official/usable/eligible
0,gpw-ref-20201127-a,17b5d4ad-0ef1-5a0e-a06e-0bb5326b889a,0.40,2020-11-27 17:05:00+01:00,2020-11-27 17:06:00+01:00,2020-11-30 09:00:00+01:00,60/57/57
1,gpw-ref-20201127-missing-negative-control,bd52f489-c099-53f7-bf4f-c0a8522ab691,0.10,2020-11-27 17:05:00+01:00,2020-11-27 17:06:00+01:00,2020-11-30 09:00:00+01:00,60/57/57


,schema_version,event_id,run_id,sequence,timestamp,account_id,security_id,...,reason_code,detail,batch_id,intent_id,target_weight,requested_quantity,accepted_quantity
0,ats.portfolio.v1,rejectionordeferredaction-54307234d5fecd220d8a...,phasec-fa439d650410376aae9e,2,2020-11-30T09:00:00+01:00,phase-c-reference-account,bd52f489-c099-53f7-bf4f-c0a8522ab691,...,missing_next_session_open,target retained as cash; existing untradeable ...,gpw-ref-20201127,gpw-ref-20201127-missing-negative-control,0.10,None,None


**Interpretation.** The 40% available target and 10% unavailable LOTOS target share official/usable/eligible counts `60/57/57`. LOTOS is rejected for `missing_next_session_open`; its weight remains explicit cash/unallocated weight. The universe never becomes “57.”

## Real trace and independent Decimal arithmetic

The first available target decides Friday after the completed bar and fills Monday at modeled open. Phase C converts canonical float64 open using `Decimal(str(value))`, applies 15 bps adverse slippage and 10 bps commission, and persists half-even rounded monetary amounts at six decimals and quantities at twelve.

In [4]:
from decimal import Decimal
from ats_portfolio.numeric import money, price, quantity

first_fill = sorted(fills, key=lambda row: row.sequence)[0]
first_order = next(row for row in orders if row.event_id == first_fill.order_id)
fill_cash = [row for row in cash_movements if row.fill_id == first_fill.event_id]
fill_position = next(row for row in position_movements if row.fill_id == first_fill.event_id)
fill_session = first_fill.modeled_market_event_ts.date()
first_valuation = next(row for row in valuations if row.security_id == first_fill.security_id and row.session_date == fill_session)
first_snapshot = next(row for row in snapshots if row.session_date == fill_session)

raw_open = Decimal(str(first_fill.raw_open_price))
requested = Decimal(str(first_order.generated_quantity))
manual_fill_price = price(raw_open * (Decimal("1") + Decimal("0.0015")))
manual_notional = money(requested * manual_fill_price)
manual_commission = money(abs(manual_notional) * Decimal("0.001"))
manual_ending_cash = money(Decimal("1000000") - manual_notional - manual_commission)
manual_market_value = money(requested * Decimal(str(first_valuation.price)))
manual_equity = money(manual_ending_cash + manual_market_value)

check = pd.DataFrame([{
    "raw_open": str(raw_open),
    "requested_and_filled_quantity": str(requested),
    "fill_price": str(manual_fill_price),
    "notional": str(manual_notional),
    "commission": str(manual_commission),
    "ending_cash": str(manual_ending_cash),
    "close_market_value": str(manual_market_value),
    "snapshot_equity": str(manual_equity),
}])
display(check)

assert manual_fill_price == first_fill.fill_price
assert requested == first_fill.quantity == fill_position.quantity_delta
assert sum((row.amount for row in fill_cash), Decimal("0")) == -(first_fill.notional + first_fill.commission)
assert manual_ending_cash == first_snapshot.cash
assert manual_equity == first_snapshot.equity

,raw_open,requested_and_filled_quantity,fill_price,notional,commission,ending_cash,close_market_value,snapshot_equity
0,25.70390000,15561.840810149433,25.74245585,400600.000000,400.600000,598999.400000,366293.052805,965292.452805


**Interpretation.** The trace independently checks fill quantity equals position movement; trade plus commission movements equal negative notional/cost; ledger cash equals snapshot cash; and cash plus marked positions equals resolved equity. The Monday close can value the portfolio only after bar completion—it was not visible for the open fill.

## Numeric policy, cash feasibility, and fail-closed invariants

All financial values are `Decimal` from text. Calculations use precision 38; persisted money/fees/notional/valuation are half-even to `0.000001`, quantities to `0.000000000001`, prices to `0.00000001`, weights/rates to `0.000000000001`.

Sells execute before buys. If buys exceed post-sale cash, one common deterministic scale—the largest affordable point on the persisted weight grid—is applied after quantity/notional/commission rounding. Residual cash remains explicit. Negative cash and short positions fail closed; cash is never silently clamped.

The accepted real run has `cash_scale=1` throughout. The one-quantum cash edge and proportional scaling are therefore evidenced by `test_scaled_buy_rounding_never_spends_more_than_available_cash` and the hand-calculated golden fixture, not by this real run.

## Missing/stale/unresolved valuation and corporate/security events

Current close is preferred. A prior revealed close is allowed only within the configured stale-session bound and records source/age/reason. If no mark is admissible, the position remains held, market value/equity become null, and the snapshot is `unresolved`; cash and quantity conservation still apply.

Stable `security_id` owns positions through ticker changes. Tested events include suspension/resumption, split, merger conversion, cash takeover, delisting without terms, identifier change, highest-known revisions, and replay-required later revisions. Adjusted bars plus explicit economic actions are rejected to prevent double counting.

**Important evidence boundary:** the accepted real run has no action/event inputs, zero corporate-action applications, only current complete valuations, and no scaled-cash case. Those behaviors are validated by synthetic tests and retained golden/state fixtures—not by the real GPW integration.

In [5]:
trade_fixture = pd.read_csv(PROJECT_ROOT / "tests" / "fixtures" / "phase_c" / "golden_trade_ledger.csv")
state_fixture = pd.read_csv(PROJECT_ROOT / "tests" / "fixtures" / "phase_c" / "golden_state_ledger.csv")
display(trade_fixture.head(8))
display(state_fixture.head(8))

,scenario,session_date,security_id,side,execution_equity,raw_open_price,requested_quantity,generated_quantity,cash_scale,fill_price,notional,commission,cash_after
0,timing_next_open,2025-01-07,A,buy,1000.00000,110.0,4.545455,4.545455,1.000000,110.165,500.750000,0.500750,498.749250
1,fractional_rebalance_initial,2025-02-03,A,buy,1000.00000,100.0,5.000000,5.000000,1.000000,100.150,500.750000,0.500750,498.749250
2,fractional_rebalance_sell,2025-02-04,A,sell,1498.74925,200.0,-3.126563,-3.126563,1.000000,199.700,-624.374718,0.624375,1122.499593
3,fractional_rebalance_buy,2025-02-04,B,buy,1498.74925,50.0,7.493746,7.493746,1.000000,50.075,375.249343,0.375249,746.875001
4,insufficient_cash_a,2025-03-03,A,buy,100.00000,100.0,0.500000,0.498752,0.997505,100.150,49.950050,0.049950,50.000000
5,insufficient_cash_b,2025-03-03,B,buy,100.00000,100.0,0.500000,0.498752,0.997505,100.150,49.950050,0.049950,0.000000


,scenario,ledger,session_date,security_id,disposition,reason_code,quantity,...,source_quantity_after,related_quantity_after,cash_delta,official_denominator,usable_count,eligible_count,unallocated_weight
0,missing_existing_open,rejections,2025-04-02,A,deferred,missing_next_session_open,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,missing_new_open,rejections,2025-04-02,B,rejected,missing_next_session_open,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,stale_close,valuations,2025-05-06,A,NaN,NaN,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,unresolved_close,valuations,2025-05-07,A,NaN,NaN,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,official_57_of_60,portfolio_snapshots,2025-06-02,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,60.0,57.0,57.0,0.43
5,split_2_for_1,corporate_action_applications,2025-01-07,AAA,NaN,NaN,NaN,...,200.0,NaN,NaN,NaN,NaN,NaN,NaN
6,reverse_split_1_for_5,corporate_action_applications,2025-01-07,AAA,NaN,NaN,NaN,...,20.0,NaN,NaN,NaN,NaN,NaN,NaN
7,merger_half_share,corporate_action_applications,2025-01-07,AAA,NaN,NaN,NaN,...,0.0,50.0,NaN,NaN,NaN,NaN,NaN


**Interpretation.** These are independently hand-calculated fixtures. Passing proves conformance to the frozen cases; it does not prove every market event, broker rule, or live-data condition is modeled.

## Accepted run, reproduction, artifacts, and logical hashes

Logical equality ignores wall-clock creation metadata but requires the same run identity and every ledger logical hash.

In [6]:
repro_manifest = LedgerRunManifest.model_validate_json(
    (PHASE_C_REPRODUCTION / "manifest.json").read_text(encoding="utf-8")
)
artifact_rows = pd.DataFrame([{
    "path": record.path,
    "bytes": record.bytes,
    "sha256": record.physical_sha256[:16],
    "logical_hash": record.logical_sha256[:16],
} for record in manifest.artifacts])
display(artifact_rows)
print({
    "same_run_id": manifest.run_id == repro_manifest.run_id,
    "same_ledger_logical_hashes": manifest.logical_hashes == repro_manifest.logical_hashes,
    "accepted_manifest_hash": manifest.manifest_hash,
    "reproduction_manifest_hash": repro_manifest.manifest_hash,
    "ending_equity_checksum": json.loads((PHASE_C_RUN / "metrics.json").read_text(encoding="utf-8"))["ending_equity_checksum"],
})
assert manifest.run_id == repro_manifest.run_id
assert manifest.logical_hashes == repro_manifest.logical_hashes

,path,bytes,sha256,logical_hash
0,config.yaml,799,c0ed4da15bb64dab,c0ed4da15bb64dab
1,environment_lock.json,370,f06f16250e25543c,f06f16250e25543c
2,inputs/intents.json,10827,4f4987d8df21fd2f,4f4987d8df21fd2f
3,inputs/phase_b_manifest.json,19138,a3ef82c6aad77b99,a3ef82c6aad77b99
4,ledgers/cash_movements.csv,3136,a1f5201484808701,4f5c76aaae0550a3
...,...,...,...,...
11,ledgers/positions.csv,2712,21d9e333933d4a95,4c0592d5fa4e3ca8
12,ledgers/rejections.csv,592,dcfebda592f5ea8b,07a254b4c3ff88cb
13,ledgers/valuations.csv,4955,c91b9f6ac86335df,448fbbd69e35d27d
14,metrics.json,272,fb8d9345169a9ff2,fb8d9345169a9ff2


{'same_run_id': True, 'same_ledger_logical_hashes': True, 'accepted_manifest_hash': 'dd31c142ef698b7c6164e3ce3de54fc67bb8aee11e1d65e032141f5fc7038ed9', 'reproduction_manifest_hash': '99f5648ddc0c8a07b25a47b7722c1b87cf54917e80543e39af42c06040298465', 'ending_equity_checksum': '1007072.062997'}


**Interpretation.** The run contains manifest/provenance, frozen intents, orders, fills, cash and position movements, valuations, snapshots, rejections, and validation evidence. It has no retained `reconciliation_report.json`; reconciliation is recomputed above. Different physical manifest hashes are expected because creation time differs, while logical ledgers match.

## What selected tests prove—and do not prove

| Test | Catches | Passing proves | Does not prove |
|---|---|---|---|
| next-open golden test | same-close leakage/cost omission | frozen next-open arithmetic | achievable market fill |
| 57/60 golden test | denominator recast | exclusions and cash weight remain visible | missing-member returns |
| stale/unresolved test | invented zero/forward value | bounded mark policy | valuation is always available |
| corporate/security state tests | quantity/cash/action/revision mistakes | named transition policy | complete corporate-action coverage |
| cash-scaling regression | one-quantum overspend/clamp | deterministic affordability | liquidity realism |
| independent validator rewrite test | hash-valid but semantically false order | target translation is reconstructed | strategy quality |
| deterministic replay | event/hash nondeterminism | same inputs yield same ledgers | future data will match |

## “I define or change a momentum signal. What happens next?”

```text
pinned Phase B canonical data
  -> point-in-time universe (`ats_data.discovery`, `ats_research.universe/panel`)
  -> feature (`ats_research.features`)
  -> label and research evaluation (`ats_research.labels`, `ats_research.diagnostics`)
  -> EXTERNAL decision/orchestration boundary (not implemented)
  -> `TargetWeightIntent` (`ats_contracts.portfolio`)
  -> Phase C simulation (`ats_portfolio.engine`, `ats_portfolio.run`)
  -> ledgers, metrics, validation, reconciliation (`storage`, `validation`)
```

Changing the feature definition changes its fingerprint and requires a new immutable research run against an explicitly pinned manifest. The platform can evaluate the feature diagnostically. It does **not** automatically choose a strategy or emit portfolio weights. An external, reviewable decision must create target-weight intents with explicit timing, denominator, exclusions, and provenance before Phase C can simulate them.

## Safe to rely on now

- Frozen intent contracts; next-eligible-open timing; Decimal ledger policy; explicit costs/movements; deterministic validation/reconciliation; official-denominator visibility.

## Usable with documented caveats

- Daily modeled open/close execution assumptions and tested stale/event policies. The real accepted run exercises only current valuations, unscaled cash, and no corporate actions.

## Not implemented or not safe to rely on

- Signal selection, automatic feature-to-weight orchestration, FX/multi-currency conversion, borrowing/margin/shorts, integer lots, dividends/rights/spinoffs, intraday/broker/live execution, optimizer, packaged-engine adapter, or investment inference from ending equity.